# Part 1 — Swin Tiny predictions (run this in a FRESH Colab runtime)
`tfswin` uses Keras 3's `keras.ops` internally, which is incompatible with the legacy
Keras 2 mode (`TF_USE_LEGACY_KERAS=1`) that LeViT needs — mixing them in one session
crashes with `AttributeError: 'KerasTensor' object has no attribute 'ndim'`.

This notebook runs Swin **alone**, in a normal (non-legacy) Keras 3 environment, and
saves its test-set and val-set prediction arrays to Drive. Run this once — it covers
**both** the Swin+LeViT+DenseNet121 and Swin+LeViT+EfficientNetB0 notebooks, since
Swin's predictions don't depend on which CNN it's paired with.

**Important:** Runtime → Disconnect and delete runtime first if you've already run
anything with `TF_USE_LEGACY_KERAS=1` in this session, then Runtime → Run all here.

## Cell 1 — Setup (installs + imports, normal Keras 3, no legacy flag)

In [1]:
!pip install tfswin --no-deps -q

import os
import shutil
import zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import tfswin  # must import before loading/rebuilding the Swin model
from tfswin import SwinTransformerTiny224
from sklearn.metrics import accuracy_score
import logging
tf.get_logger().setLevel('ERROR')
logging.getLogger('tensorflow').setLevel(logging.ERROR)

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("TF_USE_LEGACY_KERAS:", os.environ.get("TF_USE_LEGACY_KERAS"))  # should be None here
print("All imports successful!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
numpy: 2.0.2
pandas: 2.2.3
TF_USE_LEGACY_KERAS: None
All imports successful!


## Cell 2 — Drive mount

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 3 — Restore test set (images + CSV)

In [3]:
split_zip_path = '/content/drive/MyDrive/split_dataset.zip'
csv_zip_path = '/content/drive/MyDrive/thesis_dataset_csv-20260716T043638Z-1-001.zip'

if not os.path.exists('/content/test'):
    shutil.copy(split_zip_path, '/content/split_dataset.zip')
    with zipfile.ZipFile('/content/split_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('/content/')

if not os.path.exists('/content/csv_data'):
    with zipfile.ZipFile(csv_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/csv_data')

print("Test folder ready:", os.path.exists('/content/test'))

Test folder ready: True


## Cell 4 — Load test + val CSVs, fix paths

In [4]:
test_df = pd.read_csv('/content/csv_data/thesis_dataset_csv/test_data.csv')
test_df['filepath'] = test_df['filepath'].str.replace('/content/split_dataset', '/content')

val_df = pd.read_csv('/content/csv_data/thesis_dataset_csv/val_data.csv')
val_df['filepath'] = val_df['filepath'].str.replace('/content/split_dataset', '/content')

class_names = sorted(test_df['label'].unique())
num_classes = len(class_names)
label_to_index = {name: i for i, name in enumerate(class_names)}
true_labels = test_df['label'].map(label_to_index).values
val_true_labels = val_df['label'].map(label_to_index).values

print("Test samples:", len(test_df), "| Val samples:", len(val_df))
print("Classes:", class_names)

Test samples: 2538 | Val samples: 2519
Classes: ['Corn_Common_Rust', 'Corn_Gray_Leaf_Spot', 'Corn_Healthy', 'Corn_Leaf_Blight', 'Rice_Bacterial_Leaf_Blight', 'Rice_Brown_Spot', 'Rice_Healthy', 'Rice_Leaf_Blast', 'Wheat_Brown_Rust', 'Wheat_Healthy', 'Wheat_Loose_Smut', 'Wheat_Yellow_Rust']


## Cell 5 — Config + datasets (Swin needs raw uint8 pixels)

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_uint8(filepath):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8)
    return img

def make_ds(paths, load_fn):
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds_uint8 = make_ds(test_df['filepath'].values, load_uint8)
val_ds_uint8 = make_ds(val_df['filepath'].values, load_uint8)
print("Datasets ready")

Datasets ready


## Cell 6 — Rebuild Swin Tiny architecture, load trained weights

In [6]:
swin_inputs = layers.Input(shape=(224, 224, 3), dtype='uint8')
swin_base = SwinTransformerTiny224(include_top=False)
x = swin_base(swin_inputs)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
swin_outputs = layers.Dense(num_classes, activation='softmax')(x)

swin_model = models.Model(swin_inputs, swin_outputs)
swin_model.load_weights('/content/drive/MyDrive/thesis_swintiny_outputs/best_swintiny_model.keras')
print("Swin model rebuilt and weights loaded!")

177485300/177485300 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Swin model rebuilt and weights loaded!


## Cell 7 — Predict on test + val sets, sanity-check accuracy

In [7]:
pred_swin = swin_model.predict(test_ds_uint8, verbose=1)
acc_swin = accuracy_score(true_labels, np.argmax(pred_swin, axis=1))
print(f"Swin Tiny TEST accuracy: {acc_swin:.4f}")

val_pred_swin = swin_model.predict(val_ds_uint8, verbose=1)
val_acc_swin = accuracy_score(val_true_labels, np.argmax(val_pred_swin, axis=1))
print(f"Swin Tiny VAL accuracy:  {val_acc_swin:.4f}")

80/80 ━━━━━━━━━━━━━━━━━━━━ 45s 351ms/step
Swin Tiny TEST accuracy: 0.9842
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 304ms/step
Swin Tiny VAL accuracy:  0.9837


## Cell 8 — Save predictions to Drive (used by both Part 2 notebooks)

In [8]:
output_folder = '/content/drive/MyDrive/thesis_ensemble_outputs/swin_predictions_cache'
os.makedirs(output_folder, exist_ok=True)

np.save(f'{output_folder}/pred_swin.npy', pred_swin)
np.save(f'{output_folder}/val_pred_swin.npy', val_pred_swin)
np.save(f'{output_folder}/true_labels.npy', true_labels)
np.save(f'{output_folder}/val_true_labels.npy', val_true_labels)

import json
with open(f'{output_folder}/class_names.json', 'w') as f:
    json.dump(list(class_names), f)

print("Saved Swin predictions + labels + class_names to:", output_folder)
print("You can now open the Part 2 notebook (Swin+LeViT+DenseNet121 or")
print("Swin+LeViT+EfficientNetB0) in a FRESH runtime and run it directly.")

Saved Swin predictions + labels + class_names to: /content/drive/MyDrive/thesis_ensemble_outputs/swin_predictions_cache
You can now open the Part 2 notebook (Swin+LeViT+DenseNet121 or
Swin+LeViT+EfficientNetB0) in a FRESH runtime and run it directly.
